In [0]:
# Databricks notebook source
# MAGIC %md
# MAGIC # 02 -- Feature Engineering para Retreino
# MAGIC Aplica feature engineering, split temporal 60/20/20 e publica feature table no UC.
# MAGIC
# MAGIC **Entrada:** `{catalog}.silver.retrain_data_clean`
# MAGIC **Saida:** `{catalog}.gold.fraud_features_retrain`

# COMMAND ----------

%pip install databricks-feature-engineering -q

# COMMAND ----------

from pyspark.sql import functions as F
import sys

sys.path.insert(0, "/Workspace/Users/eduardolimaofficial410@gmail.com/databricks-fraud-detection/src")
from databricks_fraud_model.utils import load_config
from databricks_fraud_model.features import FEATURES_DROP, apply_feature_engineering_spark
from databricks.feature_engineering import FeatureEngineeringClient

cfg = load_config()
CATALOG = cfg["catalog"]
SOURCE = f"{CATALOG}.silver.retrain_data_clean"
FEATURE_TABLE = f"{CATALOG}.gold.fraud_features_retrain"

# COMMAND ----------

dados = spark.sql(f"SELECT * FROM {SOURCE}")
dados = dados.drop(*[c for c in FEATURES_DROP if c in dados.columns])
dados_fe = apply_feature_engineering_spark(dados)

# Split temporal 60/20/20
q60, q80 = dados_fe.approxQuantile('TransactionDT', [0.60, 0.80], 0.01)

full_df = (
    dados_fe.filter(F.col('TransactionDT') <= q60).withColumn("split", F.lit("train"))
    .unionByName(dados_fe.filter((F.col('TransactionDT') > q60) & (F.col('TransactionDT') <= q80)).withColumn("split", F.lit("test")))
    .unionByName(dados_fe.filter(F.col('TransactionDT') > q80).withColumn("split", F.lit("val")))
    .filter(F.col("TransactionID").isNotNull())
    .withColumn("TransactionDT", F.from_unixtime(F.col("TransactionDT")).cast("timestamp"))
)

spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.gold")
spark.sql(f"DROP TABLE IF EXISTS {FEATURE_TABLE}")

fe = FeatureEngineeringClient()
fe.create_table(
    name=FEATURE_TABLE,
    primary_keys=["TransactionID", "TransactionDT"],
    timeseries_column="TransactionDT",
    df=full_df, schema=full_df.schema,
    description="Feature table para retreino Champion/Challenger",
)

display(spark.sql(f"SELECT split, count(*) as n FROM {FEATURE_TABLE} GROUP BY split"))